<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip__Interativo_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BIBLIOTECA

In [9]:
### Rodar essa célula somente uma vez ###
# Delete a # na linha abaixo, execute e coloque de volta a #

#!pip install plotly

In [10]:
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

In [11]:
pd.options.display.float_format = '{:,.4f}'.format

### ARQUIVO CSV

In [12]:
# Parametros de entrada
filename = 'quotedata.csv'

# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else: # Gamma is same for calls and puts. This is just to cross-check
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [13]:
# Isso assume que o formato do arquivo CBOE não foi editado, ou seja, a tabela começa na linha 4
optionsFile = open(filename)
optionsFileData = optionsFile.readlines()
optionsFile.close()

In [14]:
# Extraindo SPX spot
spotLine = optionsFileData[1]
spotPrice = float(spotLine.split('Last:')[1].split(',')[0])
fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

In [15]:
# Extraindo a data de hoje
dateLine = optionsFileData[2]
todayDate = dateLine.split('Date: ')[1].split(',')
monthDay = todayDate[0].split(' ')

In [16]:
if len(monthDay) == 2:
    year = int(monthDay[4])
    month = monthDay[2]
    day = int(monthDay[0])
else:
    if monthDay[2].isdigit():
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])
    else:
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])


# criar um dicionário para mapear os nomes dos meses em português para os equivalentes em inglês
nomes_meses = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extrair o nome do mês da string de entrada
nome_mes_pt = monthDay[2]

# converter o nome do mês para inglês usando o dicionário
nome_mes_en = nomes_meses[nome_mes_pt]

# converter o nome do mês para o número correspondente (por exemplo, 'March' -> 3)
num_mes = datetime.strptime(nome_mes_en, '%B').month

# criar o objeto datetime
todayDate = datetime(year=year, month=num_mes, day=day)

In [17]:
# create a dictionary to map Portuguese month names to English month names
month_names = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extract the month name from the input string
month_name_pt = monthDay[2]

# convert the month name to English using the dictionary
month_name_en = month_names[month_name_pt]

# convert the month name to its corresponding number (e.g., 'March' -> 3)
month_number = datetime.strptime(month_name_en, '%B').month

# create the datetime object
todayDate = datetime(year=year, month=month_number, day=day)

In [18]:
# Get SPX Options Data
df = pd.read_csv(filename, sep=",", header=None, skiprows=4)
df.columns = ['ExpirationDate','Calls','CallLastSale','CallNet','CallBid','CallAsk','CallVol',
              'CallIV','CallDelta','CallGamma','CallOpenInt','StrikePrice','Puts','PutLastSale',
              'PutNet','PutBid','PutAsk','PutVol','PutIV','PutDelta','PutGamma','PutOpenInt']


df['ExpirationDate'] = pd.to_datetime(df['ExpirationDate'], format='%a %b %d %Y')
df['ExpirationDate'] = df['ExpirationDate'] + timedelta(hours=16)
df['StrikePrice'] = df['StrikePrice'].astype(float)
df['CallIV'] = df['CallIV'].astype(float)
df['PutIV'] = df['PutIV'].astype(float)
df['CallGamma'] = df['CallGamma'].astype(float)
df['PutGamma'] = df['PutGamma'].astype(float)
df['CallOpenInt'] = df['CallOpenInt'].astype(float)
df['PutOpenInt'] = df['PutOpenInt'].astype(float)

### GAMMA (CHART 1, 2 E 3)

In [19]:
# ---=== CALCULATE SPOT GAMMA ===---
# Gamma Exposure = Unit Gamma * Open Interest * Contract Size * Spot Price
# To further convert into 'per 1% move' quantity, multiply by 1% of spotPrice
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [20]:
# Chart 1: Absolute Gamma Exposure
# define os dados
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

# cria um gráfico de barras
fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Gamma Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig.show()

In [21]:
# CALL WALL E PUT WALL - CHART 1
call_oi_wall_strike = df.loc[df['CallOpenInt'].idxmax()]['StrikePrice']
call_oi_wall_value = df['CallOpenInt'].max()

# Find the strike with the highest Put Open Interest and its value
put_oi_wall_strike = df.loc[df['PutOpenInt'].idxmax()]['StrikePrice']
put_oi_wall_value = df['PutOpenInt'].max()

# Find the strike with the highest Call Volume and its value
call_vol_wall_strike = df.loc[df['CallVol'].idxmax()]['StrikePrice']
call_vol_wall_value = df['CallVol'].max()

# Find the strike with the highest Put Volume and its value
put_vol_wall_strike = df.loc[df['PutVol'].idxmax()]['StrikePrice']
put_vol_wall_value = df['PutVol'].max()

# Find the top 5 strikes with the highest Call Open Interest
top5_call_oi = df.nlargest(5, 'CallOpenInt')[['StrikePrice', 'CallOpenInt']]

# Find the top 5 strikes with the highest Put Open Interest
top5_put_oi = df.nlargest(5, 'PutOpenInt')[['StrikePrice', 'PutOpenInt']]


print("--- Dados de Open Interest e Volume para Call/Put Walls ---")

print("\nParedes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\nParedes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\nTop 5 Calls por Open Interest:")
for index, row in top5_call_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")

print("\nTop 5 Puts por Open Interest:")
for index, row in top5_put_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")

--- Dados de Open Interest e Volume para Call/Put Walls ---

Paredes por Open Interest:
  Call Wall (OI): 23106 at strike 7000
  Put Wall (OI): 37042 at strike 5420

Paredes por Volume:
  Call Wall (Vol): 39010 at strike 6880
  Put Wall (Vol): 19313 at strike 6850

Top 5 Calls por Open Interest:
  Strike 7000: 23106
  Strike 7200: 21357
  Strike 6950: 19314
  Strike 6940: 12891
  Strike 6875: 12382

Top 5 Puts por Open Interest:
  Strike 5420: 37042
  Strike 5320: 36847
  Strike 6000: 28212
  Strike 5100: 23599
  Strike 5500: 23099


In [41]:
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the top 9 smallest gamma values (most negative)
smallest_gamma = dfAgg_sorted.head(9)
print("Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
# Iterate through smallest_gamma in descending order of TotalGamma (already sorted ascending, so reverse)
for index, row in smallest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Get the top 9 largest gamma values (most positive)
largest_gamma = dfAgg_sorted.tail(9)
print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
# Iterate through largest_gamma in descending order of TotalGamma (already sorted ascending, so reverse)
for index, row in largest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Add the Zero Gamma (Gamma Flip) point
print(f"\nZero Gamma (Gamma Flip): {zeroGamma:.0f}")

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")


# --- Find the single largest strike by Volume and Open Interest (Overall) ---

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
largest_call_vol_strike = df.loc[df['CallVol'].idxmax()]
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
largest_put_vol_strike = df.loc[df['PutVol'].idxmax()]
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
largest_call_oi_strike = df.loc[df['CallOpenInt'].idxmax()]
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
largest_put_oi_strike = df.loc[df['PutOpenInt'].idxmax()]
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")

Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):
  Strike 6680: Type: P, Delta: 327.2409 M, Gamma: -0.1897 Bn
  Strike 6550: Type: P, Delta: 278.4858 M, Gamma: -0.2141 Bn
  Strike 6500: Type: P, Delta: 4801.2301 M, Gamma: -0.2303 Bn
  Strike 6675: Type: P, Delta: 1893.5576 M, Gamma: -0.2796 Bn
  Strike 6730: Type: P, Delta: 1225.4274 M, Gamma: -0.2976 Bn
  Strike 6700: Type: P, Delta: 5281.8723 M, Gamma: -0.3153 Bn
  Strike 6720: Type: P, Delta: 1020.2277 M, Gamma: -0.3208 Bn
  Strike 6740: Type: P, Delta: 1093.5786 M, Gamma: -0.3801 Bn
  Strike 6600: Type: P, Delta: 1968.1078 M, Gamma: -0.4185 Bn

Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):
  Strike 6850: Type: C, Delta: 7497.5661 M, Gamma: 7.5771 Bn
  Strike 6875: Type: C, Delta: 2597.5389 M, Gamma: 7.0814 Bn
  Strike 6860: Type: C, Delta: 3178.8961 M, Gamma: 6.4448 Bn
  Strike 6825: Type: C, Delta: 6826.2678 M, Gamma: 4.6002 Bn
  Strike 6870: Type: C, Delta: 1603.8816 M, Gamma: 4.4433 Bn
  Strike 688

In [24]:
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])
chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")
fig.add_shape(dict(type="line", x0=spotPrice, y0=0, x1=spotPrice, y1=max(dfAgg['CallGEX'].to_numpy() / 10**9), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)

fig.show()


In [25]:
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 6850, Call GEX: 7.8293 Bn, Put GEX: -0.2522 Bn
  GEX Level 2: Strike 6875, Call GEX: 7.2299 Bn, Put GEX: -0.1485 Bn
  GEX Level 3: Strike 6860, Call GEX: 6.7913 Bn, Put GEX: -0.3465 Bn
  GEX Level 4: Strike 6825, Call GEX: 4.8768 Bn, Put GEX: -0.2767 Bn
  GEX Level 5: Strike 6800, Call GEX: 3.6859 Bn, Put GEX: -1.2853 Bn
  GEX Level 6: Strike 6870, Call GEX: 4.5108 Bn, Put GEX: -0.0675 Bn


In [26]:
# ---=== CALCULATE GAMMA PROFILE ===---
levels = np.linspace(fromStrike, toStrike, 60)

# For 0DTE options, I'm setting DTE = 1 day, otherwise they get excluded
df['daysTillExp'] = [1/262 if (np.busday_count(todayDate.date(), x.date())) == 0 \
                           else np.busday_count(todayDate.date(), x.date())/262 for x in df.ExpirationDate]

nextExpiry = df['ExpirationDate'].min()

df['IsThirdFriday'] = [isThirdFriday(x) for x in df.ExpirationDate]
thirdFridays = df.loc[df['IsThirdFriday'] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

In [27]:
# For each spot level, calc gamma exposure at that point
for level in levels:
    df['callGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df['putGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma.append(df['callGammaEx'].sum() - df['putGammaEx'].sum())

    exNxt = df.loc[df['ExpirationDate'] != nextExpiry]
    totalGammaExNext.append(exNxt['callGammaEx'].sum() - exNxt['putGammaEx'].sum())

    exFri = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalGammaExFri.append(exFri['callGammaEx'].sum() - exFri['putGammaEx'].sum())

totalGamma = np.array(totalGamma) / 10**9
totalGammaExNext = np.array(totalGammaExNext) / 10**9
totalGammaExFri = np.array(totalGammaExFri) / 10**9

In [28]:
# Find Gamma Flip Point
zeroCrossIdx = np.where(np.diff(np.sign(totalGamma)))[0]

negGamma = totalGamma[zeroCrossIdx]
posGamma = totalGamma[zeroCrossIdx+1]
negStrike = levels[zeroCrossIdx]
posStrike = levels[zeroCrossIdx+1]

zeroGamma = posStrike - ((posStrike - negStrike) * posGamma/(posGamma-negGamma))
zeroGamma = zeroGamma[0]

In [29]:
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(title=chartTitle, xaxis_title='Index Price', yaxis_title='Gamma Exposure ($ billions/1% move)')
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))

fig.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalGamma),
        x1=spotPrice,
        y1=max(totalGamma),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

fig.add_shape(
    dict(
        type="line",
        x0=zeroGamma,
        y0=min(totalGamma),
        x1=zeroGamma,
        y1=max(totalGamma),
        line=dict(color="green", width=1.5),
        name="Gamma Flip: " + str("{:,.0f}".format(zeroGamma))
    )
)

fig.update_xaxes(range=[fromStrike, toStrike])
fig.update_yaxes(range=[min(totalGamma), max(totalGamma)])

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[min(totalGamma), min(totalGamma), min(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="red",
        opacity=0.1,
        showlegend=False,
        name="Negative Gamma"
    )
)

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[max(totalGamma), max(totalGamma), max(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="green",
        opacity=0.1,
        showlegend=False,
        name="Positive Gamma"
    )
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1400,
    height=700
)

fig.show()

In [30]:
# DADOS CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")

# Vol Trigger (value at zero gamma cross) for Ex-Next Monthly Expiry
vol_trigger_value_at_flip_exfri = np.interp(zeroGamma, levels, totalGammaExFri)
print(f"Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): {vol_trigger_value_at_flip_exfri:.4f} at strike {zeroGamma:.0f}")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

Gamma Flip (Ex-Next Monthly Expiry): 7.6397 at strike 6781
Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): 0.0000 at strike 6762
Max Gamma Positivo (Ex-Next Monthly Expiry): 58.7365 at strike 6874
Min Gamma Negativo (Ex-Next Monthly Expiry): -30.8880 at strike 6549


In [44]:
# Consolidating results from CHART 1, CHART 2, and CHART 3
# Requires dfAgg_delta from cell 5f01c222
# Requires levels, totalGammaExFri, zeroGamma, max_gamma_positive_strike, min_gamma_negative_strike from previous cells
# Requires levels_delta, totalDeltaExFri, spotPrice from previous cells


print("="*80)
print("📊 RESUMO CONSOLIDADO DOS DADOS")
print("="*80)

# --- DADOS DO CHART 1 ---
print("\n--- DADOS DO CHART 1 (Gamma Exposure) ---")
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the top 9 smallest gamma values (most negative)
smallest_gamma = dfAgg_sorted.head(9)
print("Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Get the top 9 largest gamma values (most positive)
largest_gamma = dfAgg_sorted.tail(9)
print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
for index, row in largest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Add the Zero Gamma (Gamma Flip) point
print(f"\nZero Gamma (Gamma Flip): {zeroGamma:.0f}")

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

# --- Dados de Open Interest e Volume para Call/Put Walls (from cell e8b1a91b) ---
print("\n--- Dados de Open Interest e Volume ---")

# Find the strike with the highest Call Open Interest and its value
call_oi_wall_strike = df.loc[df['CallOpenInt'].idxmax()]['StrikePrice']
call_oi_wall_value = df['CallOpenInt'].max()

# Find the strike with the highest Put Open Interest and its value
put_oi_wall_strike = df.loc[df['PutOpenInt'].idxmax()]['StrikePrice']
put_oi_wall_value = df['PutOpenInt'].max()

# Find the strike with the highest Call Volume and its value
call_vol_wall_strike = df.loc[df['CallVol'].idxmax()]['StrikePrice']
call_vol_wall_value = df['CallVol'].max()

# Find the strike with the highest Put Volume and its value
put_vol_wall_strike = df.loc[df['PutVol'].idxmax()]['StrikePrice']
put_vol_wall_value = df['PutVol'].max()

# Find the top 5 strikes with the highest Call Open Interest
top5_call_oi = df.nlargest(5, 'CallOpenInt')[['StrikePrice', 'CallOpenInt']]

# Find the top 5 strikes with the highest Put Open Interest
top5_put_oi = df.nlargest(5, 'PutOpenInt')[['StrikePrice', 'PutOpenInt']]

print("\nParedes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\nParedes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\nTop 5 Calls por Open Interest:")
for index, row in top5_call_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")

print("\nTop 5 Puts por Open Interest:")
for index, row in top5_put_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")

# --- DADOS CHART 2 (GEX Levels) ---
print("\n--- DADOS CHART 2 (GEX Levels) ---")
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

# --- DADOS CHART 3 (Gamma Profile and Delta Flip) ---
print("\n--- DADOS CHART 3 (Gamma Profile & Delta Flip) ---")
# Gamma Flip (based on zeroGamma)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")

# Vol Trigger (value at zero gamma cross) for Ex-Next Monthly Expiry (based on zeroGamma)
vol_trigger_value_at_flip_exfri = np.interp(zeroGamma, levels, totalGammaExFri)
print(f"Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): {vol_trigger_value_at_flip_exfri:.4f} at strike {zeroGamma:.0f}")

# Max Gamma Positivo (Ex-Next Monthly Expiry)
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")

# Min Gamma Negativo (Ex-Next Monthly Expiry)
min_gamma_negative_value = np.min(totalGammaExFri) # Using totalDeltaExFri by mistake? Should be totalGammaExFri
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index] # Corrected variable name to min_gamma_negative_strike
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

# Delta Flip (Delta at Spot Price for Ex-Next Monthly Expiry line)
delta_at_spot_exfri = np.interp(spotPrice, levels_delta, totalDeltaExFri)
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_exfri:.4f} Million at strike {spotPrice:.0f}")

📊 RESUMO CONSOLIDADO DOS DADOS

--- DADOS DO CHART 1 (Gamma Exposure) ---
Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):
  Strike 6680: Type: P, Delta: 327.2409 M, Gamma: -0.1897 Bn
  Strike 6550: Type: P, Delta: 278.4858 M, Gamma: -0.2141 Bn
  Strike 6500: Type: P, Delta: 4801.2301 M, Gamma: -0.2303 Bn
  Strike 6675: Type: P, Delta: 1893.5576 M, Gamma: -0.2796 Bn
  Strike 6730: Type: P, Delta: 1225.4274 M, Gamma: -0.2976 Bn
  Strike 6700: Type: P, Delta: 5281.8723 M, Gamma: -0.3153 Bn
  Strike 6720: Type: P, Delta: 1020.2277 M, Gamma: -0.3208 Bn
  Strike 6740: Type: P, Delta: 1093.5786 M, Gamma: -0.3801 Bn
  Strike 6600: Type: P, Delta: 1968.1078 M, Gamma: -0.4185 Bn

Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):
  Strike 6850: Type: C, Delta: 7497.5661 M, Gamma: 7.5771 Bn
  Strike 6875: Type: C, Delta: 2597.5389 M, Gamma: 7.0814 Bn
  Strike 6860: Type: C, Delta: 3178.8961 M, Gamma: 6.4448 Bn
  Strike 6825: Type: C, Delta: 6826.2678 M, Gamma: 4.6002 Bn

### DELTA (CHART 4,5 E 6)

In [43]:
# ---=== CALCULATE SPOT DELTA ===---
# Delta Exposure = Unit Delta * Open Interest * Contract Size * Spot Price
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice

# Total Delta considers the sign of delta for calls and puts
df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6 # Converting to millions for better scaling

dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions

# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [37]:
# Chart 4: Absolute Delta Exposure
# define os dados
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

# cria um gráfico de barras
fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Delta Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta4.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig_delta4.show()

In [45]:
# --- DADOS DO CHART 4 (Delta Exposure) ---
print("="*80)
print("📊 DADOS DO CHART 4 (Delta Exposure)")
print("="*80)

# Requires dfAgg_delta from cell 5f01c222
# Requires totalDelta from cell 6143f354
# Requires zeroDelta from cell 9e43625b
# Requires df from cell I3o4YVMQogB_

dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Get the top 9 smallest delta exposure values (most negative)
smallest_delta_exposure = dfAgg_delta_sorted.head(9)
print("\nTop 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Get the top 9 largest delta exposure values (most positive)
largest_delta_exposure = dfAgg_delta_sorted.tail(9)
print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for index, row in largest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Add the Zero Delta (Delta Flip) point
# zeroDelta is calculated in cell 9e43625b
if zeroDelta is not None:
    print(f"\nZero Delta (Delta Flip): {zeroDelta:.0f}")
else:
    print("\nZero Delta (Delta Flip): Não encontrado")

# Print the total delta exposure
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")


# --- Strikes com Maior Volume e Open Interest (Geral) ---
# Referencing variables already calculated in cell e8b1a91b

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
# largest_call_vol_strike is from cell e8b1a91b
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
# largest_put_vol_strike is from cell e8b1a91b
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
# largest_call_oi_strike is from cell e8b1a91b
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
# largest_put_oi_strike is from cell e8b1a91b
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")

📊 DADOS DO CHART 4 (Delta Exposure)

Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):
  Strike 6310: Type: P, Gamma: -0.0030 Bn, Delta: -8.9770 M
  Strike 5320: Type: P, Gamma: 0.0000 Bn, Delta: -10.0978 M
  Strike 5420: Type: P, Gamma: 0.0000 Bn, Delta: -10.4348 M
  Strike 6555: Type: P, Gamma: -0.0907 Bn, Delta: -11.6484 M
  Strike 6170: Type: P, Gamma: -0.0257 Bn, Delta: -16.9877 M
  Strike 6085: Type: P, Gamma: 0.0000 Bn, Delta: -19.0456 M
  Strike 6185: Type: P, Gamma: -0.0304 Bn, Delta: -23.1633 M
  Strike 7025: Type: P, Gamma: 0.0807 Bn, Delta: -159.9683 M
  Strike 7000: Type: P, Gamma: 0.9135 Bn, Delta: -4711.5253 M

Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):
  Strike 6800: Type: C, Gamma: 2.4006 Bn, Delta: 9988.3179 M
  Strike 6750: Type: C, Gamma: 0.0665 Bn, Delta: 9715.6915 M
  Strike 6650: Type: C, Gamma: 0.0588 Bn, Delta: 8673.4106 M
  Strike 6850: Type: C, Gamma: 7.5771 Bn, Delta: 7497.5661 M
  Strike 6000: Type: C, Gamma: 0.0000 Bn, Delt

In [47]:
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])
chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")
fig_delta5.add_shape(dict(type="line", x0=spotPrice, y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6), x1=spotPrice, y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta5.update_layout(
    width=1750,
    height=800
)

fig_delta5.show()

In [46]:
# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions

In [48]:
# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [49]:
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(title=chartTitle_delta6, xaxis_title='Index Price', yaxis_title='Delta Exposure ($ millions/1% move)')
fig_delta6.update_layout(title_text=chartTitle_delta6, title_font=dict(size=20, family="Arial Black"))

fig_delta6.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalDelta),
        x1=spotPrice,
        y1=max(totalDelta),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

# Add the Delta Flip line only if zeroDelta is not None
if zeroDelta is not None:
    fig_delta6.add_shape(
        dict(
            type="line",
            x0=zeroDelta,
            y0=min(totalDelta),
            x1=zeroDelta,
            y1=max(totalDelta),
            line=dict(color="green", width=1.5),
            # Format zeroDelta as a scalar
            name="Delta Flip: " + str("{:,.0f}".format(float(zeroDelta)))
        )
    )


fig_delta6.update_xaxes(range=[fromStrike, toStrike])
fig_delta6.update_yaxes(range=[min(totalDelta), max(totalDelta)])

# Adding shaded areas for positive and negative delta
# Adjust shaded areas to account for potential None zeroDelta
if zeroDelta is not None:
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[min(totalDelta), min(totalDelta), min(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="red",
            opacity=0.1,
            showlegend=False,
            name="Negative Delta"
        )
    )

    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[max(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="green",
            opacity=0.1,
            showlegend=False,
            name="Positive Delta"
        )
    )
else:
     # If no zeroDelta, the entire range is either positive or negative
    fill_color = 'green' if totalDelta[0] >= 0 else 'red'
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, toStrike, toStrike, fromStrike],
            y=[min(totalDelta), min(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor=fill_color,
            opacity=0.1,
            showlegend=False,
            name="Delta Region"
        )
    )


# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta6.update_layout(
    width=1400,
    height=700
)

fig_delta6.show()

In [50]:
# Consolidating results from CHART 4, CHART 5, and CHART 6
print("--- DADOS CHART 4 ---")
# Requires dfAgg_delta from cell 5f01c222
# Requires df from cell I3o4YVMQogB_
# Requires zeroDelta from cell 9e43625b
# Requires largest_call_vol_strike, largest_put_vol_strike, largest_call_oi_strike, largest_put_oi_strike from cell e8b1a91b


dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Get the top 9 smallest delta exposure values (most negative)
smallest_delta_exposure = dfAgg_delta_sorted.head(9)
print("Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Get the top 9 largest delta exposure values (most positive)
largest_delta_exposure = dfAgg_delta_sorted.tail(9)
print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for index, row in largest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Add the Zero Delta (Delta Flip) point (based on zero cross)
if zeroDelta is not None:
    print(f"\nZero Delta (Delta Flip - Zero Cross): {zeroDelta:.0f}")
else:
    print("\nZero Delta (Delta Flip - Zero Cross): Não encontrado")


# Print the total delta exposure
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

# --- Strikes com Maior Volume e Open Interest (Geral) ---
# Referencing variables already calculated in cell e8b1a91b

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
# largest_call_vol_strike is from cell e8b1a91b
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
# largest_put_vol_strike is from cell e8b1a91b
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
# largest_call_oi_strike is from cell e8b1a91b
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
# largest_put_oi_strike is from cell e8b1a91b
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")


print("\n--- DADOS CHART 5 ---")
# DATA FOR CHART 5 (Absolute Delta Exposure by Calls and Puts)
# Calculate the absolute sum of Call and Put Delta Exposure for each strike
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX
dex_levels = dfAgg_delta_sorted_dex.head(6)

print("Top 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure):")
for i in range(len(dex_levels)):
    strike_dex = dex_levels.index[i]
    call_dex = dex_levels.iloc[i]['CallDEX'] / 10**6
    put_dex = dex_levels.iloc[i]['PutDEX'] / 10**6
    print(f"  DEX Level {i+1}: Strike {strike_dex:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")

print("\n--- DADOS CHART 6 ---")
# DATA FOR CHART 6 (Delta Exposure Profile)
# Find Delta Flip (first value above the central green line) for Ex-Next Monthly Expiry Delta Profile
# This assumes totalDeltaExFri is the data for the 'Ex-Next Monthly Expiry' line
# delta_flip_index_exfri = np.where(totalDeltaExFri > 0)[0][0] if np.any(totalDeltaExFri > 0) else None
# if delta_flip_index_exfri is not None:
#     delta_flip_strike_exfri = levels_delta[delta_flip_index_exfri]
#     delta_flip_value_exfri = totalDeltaExFri[delta_flip_index_exfri]
#     print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_flip_value_exfri:.4f} at strike {delta_flip_strike_exfri:.0f}")
# else:
#     print("No Delta Flip found (Ex-Next Monthly Expiry).")

# Delta Vol Trigger (value at zero delta cross) for Ex-Next Monthly Expiry
# Use the zeroDelta calculated earlier for the Delta Flip point
# if zeroDelta is not None:
#     delta_vol_trigger_value_at_flip_exfri = np.interp(zeroDelta, levels_delta, totalDeltaExFri)
#     print(f"Delta Vol Trigger (Delta Flip Point, Ex-Next Monthly Expiry): {delta_vol_trigger_value_at_flip_exfri:.4f} at strike {zeroDelta:.0f}")
# else:
#     print("Delta Vol Trigger not found (no Delta Flip point).")

# Max Positive Delta (Ex-Next Monthly Expiry)
max_delta_positive_value_exfri = np.max(totalDeltaExFri)
max_delta_positive_index_exfri = np.argmax(totalDeltaExFri)
max_delta_positive_strike_exfri = levels_delta[max_delta_positive_index_exfri]
print(f"Max Delta Positivo (Ex-Next Monthly Expiry): {max_delta_positive_value_exfri:.4f} at strike {max_delta_positive_strike_exfri:.0f}")

# Min Negative Delta (Ex-Next Monthly Expiry)
min_delta_negative_value_exfri = np.min(totalDeltaExFri)
min_delta_negative_index_exfri = np.argmin(totalDeltaExFri)
min_delta_negative_strike_exfri = levels_delta[min_delta_negative_index_exfri]
print(f"Min Delta Negativo (Ex-Next Monthly Expiry): {min_delta_negative_value_exfri:.4f} at strike {min_delta_negative_strike_exfri:.0f}")

# Find the Delta Exposure value at the Spot Price for the Ex-Next Monthly Expiry line and label as Delta Flip
# delta_at_spot_exfri is already calculated above
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_exfri:.4f} Million at strike {spotPrice:.0f}")

--- DADOS CHART 4 ---
Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):
  Strike 6310: Type: P, Gamma: -0.0030 Bn, Delta: -8.9770 M
  Strike 5320: Type: P, Gamma: 0.0000 Bn, Delta: -10.0978 M
  Strike 5420: Type: P, Gamma: 0.0000 Bn, Delta: -10.4348 M
  Strike 6555: Type: P, Gamma: -0.0907 Bn, Delta: -11.6484 M
  Strike 6170: Type: P, Gamma: -0.0257 Bn, Delta: -16.9877 M
  Strike 6085: Type: P, Gamma: 0.0000 Bn, Delta: -19.0456 M
  Strike 6185: Type: P, Gamma: -0.0304 Bn, Delta: -23.1633 M
  Strike 7025: Type: P, Gamma: 0.0807 Bn, Delta: -159.9683 M
  Strike 7000: Type: P, Gamma: 0.9135 Bn, Delta: -4711.5253 M

Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):
  Strike 6800: Type: C, Gamma: 2.4006 Bn, Delta: 9988.3179 M
  Strike 6750: Type: C, Gamma: 0.0665 Bn, Delta: 9715.6915 M
  Strike 6650: Type: C, Gamma: 0.0588 Bn, Delta: 8673.4106 M
  Strike 6850: Type: C, Gamma: 7.5771 Bn, Delta: 7497.5661 M
  Strike 6000: Type: C, Gamma: 0.0000 Bn, Delta: 7226.1410 M


### ARQUIVO PARA TRADING VIEW

In [58]:
# ==================== CÉLULA FINAL DO NOTEBOOK ====================
# Cole esta célula no final do seu notebook Jupyter/Colab
# Ela irá gerar UMA ÚNICA LINHA para atualizar tudo de uma vez

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA/DELTA FLIP (VERSÃO COMPLETA)") # Updated title for clarity
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================
# Referencing variables calculated in preceding cells (e.g., from 035f55f4, be13ea2c, Egb8foZPG3zj, etc.)

# Dados gerais
spot_price = spotPrice # from fvEJmuujccxO
total_gamma = df['TotalGamma'].sum() # from I3o4YVMQogB_
total_delta = df['TotalDelta'].sum() # from I3o4YVMQogB_
update_date = todayDate.strftime('%d %b %Y 00:00') # from fvEJmuujccxO

# CHART 1 - Spot Gamma Levels (from Egb8foZPG3zj and be13ea2c)
# smallest_gamma and largest_gamma are from Egb8foZPG3zj and be13ea2c
# Using the sorted dataframes to get the top/bottom strikes by gamma exposure
dfAgg_sorted_gamma = dfAgg.sort_values(by='TotalGamma')

# Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente)
smallest_gamma_9 = dfAgg_sorted_gamma.head(9)
chart1_neg_gamma_data = []
for index, row in smallest_gamma_9.iloc[::-1].iterrows():
    chart1_neg_gamma_data.append({'strike': index, 'gamma': row['TotalGamma']})

# Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente)
largest_gamma_9 = dfAgg_sorted_gamma.tail(9)
chart1_pos_gamma_data = []
for index, row in largest_gamma_9.iloc[::-1].iterrows():
     chart1_pos_gamma_data.append({'strike': index, 'gamma': row['TotalGamma']})

# Zero Gamma (Gamma Flip) point (from REAlEsxDtvng and be13ea2c)
# zeroGamma is from REAlEsxDtvng and be13ea2c
chart1_zero_gamma_strike = zeroGamma


# Open Interest and Volume Walls (from e8b1a91b and be13ea2c)
# Using the variables already calculated in e8b1a91b and be13ea2c
chart1_call_oi_wall_strike = call_oi_wall_strike
chart1_call_oi_wall_value = call_oi_wall_value
chart1_put_oi_wall_strike = put_oi_wall_strike
chart1_put_oi_wall_value = put_oi_wall_value

chart1_call_vol_wall_strike = call_vol_wall_strike
chart1_call_vol_wall_value = call_vol_wall_value
chart1_put_vol_wall_strike = put_vol_wall_strike
chart1_put_vol_wall_value = put_vol_wall_value

# Top 5 Calls por Open Interest (from e8b1a91b and be13ea2c)
# top5_call_oi is from e8b1a91b and be13ea2c
chart1_top5_call_oi_data = top5_call_oi.to_dict('records')

# Top 5 Puts por Open Interest (from e8b1a91b and be13ea2c)
# top5_put_oi is from e8b1a91b and be13ea2c
chart1_top5_put_oi_data = top5_put_oi.to_dict('records')

# Strikes com Maior Volume e Open Interest (Geral) (from e8b1a91b and be13ea2c)
# largest_call_vol_strike, largest_put_vol_strike, largest_call_oi_strike, largest_put_oi_strike are from e8b1a91b and be13ea2c
chart1_largest_call_vol_strike = largest_call_vol_strike['StrikePrice']
chart1_largest_call_vol_value = largest_call_vol_strike['CallVol']
chart1_largest_put_vol_strike = largest_put_vol_strike['StrikePrice']
chart1_largest_put_vol_value = largest_put_vol_strike['PutVol']
chart1_largest_call_oi_strike = largest_call_oi_strike['StrikePrice']
chart1_largest_call_oi_value = largest_call_oi_strike['CallOpenInt']
chart1_largest_put_oi_strike = largest_put_oi_strike['StrikePrice']
chart1_largest_put_oi_value = largest_put_oi_strike['PutOpenInt']


# CHART 2 - GEX Levels (from d0765a4d-2f3d-449c-a61b-9ef63ffb85ac and be13ea2c)
# gex_levels is from d0765a4d-2f3d-449c-a61b-9ef63ffb85ac and be13ea2c
chart2_gex_levels_data = []
for i in range(min(6, len(gex_levels))):
    chart2_gex_levels_data.append({
        'strike': gex_levels.index[i],
        'call_gex': gex_levels.iloc[i]['CallGEX'] / 10**9, # Convert to billions
        'put_gex': gex_levels.iloc[i]['PutGEX'] / 10**9   # Convert to billions
    })

# CHART 3 - Gamma Profile points (from 0eb2b7f4 and be13ea2c)
# gamma_flip_value, gamma_flip_strike, max_gamma_positive_value, max_gamma_positive_strike,
# min_gamma_negative_value, min_gamma_negative_strike are from 0eb2b7f4 and be13ea2c
chart3_gamma_flip_strike = gamma_flip_strike
chart3_gamma_flip_value = gamma_flip_value
chart3_vol_trigger_strike = zeroGamma # Vol Trigger strike is the Gamma Flip point
chart3_vol_trigger_value = vol_trigger_value_at_flip_exfri # Vol Trigger value at Gamma Flip point
chart3_max_pos_strike = max_gamma_positive_strike
chart3_max_pos_value = max_gamma_positive_value
chart3_min_neg_strike = min_gamma_negative_strike
chart3_min_neg_value = min_gamma_negative_value

# Delta Flip from Chart 3 (based on Delta at Spot Price for Ex-Next Monthly Expiry line) - Label as Estrutura
chart3_delta_flip_estrutura_strike = spotPrice # Delta Flip (Estrutura) strike is the Spot Price
chart3_delta_flip_estrutura_value = np.interp(spotPrice, levels_delta, totalDeltaExFri) # Interpolate Delta value at Spot Price

# CHART 4 - Delta Exposure points (from 035f55f4)
# dfAgg_delta_sorted is from 035f55f4
dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente)
smallest_delta_exposure_9 = dfAgg_delta_sorted.head(9)
chart4_neg_delta_data = []
for index, row in smallest_delta_exposure_9.iloc[::-1].iterrows():
    chart4_neg_delta_data.append({'strike': index, 'delta': row['TotalDelta']})

# Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente)
largest_delta_exposure_9 = dfAgg_delta_sorted.tail(9)
chart4_pos_delta_data = []
for index, row in largest_delta_exposure_9.iloc[::-1].iterrows():
    chart4_pos_delta_data.append({'strike': index, 'delta': row['TotalDelta']})

# Zero Delta (Delta Flip) point (based on zero cross from 9e43625b and 035f55f4)
# zeroDelta is from 9e43625b and 035f55f4
chart4_zero_delta_strike = zeroDelta if zeroDelta is not None else 0 # Use 0 if not found


# CHART 5 - DEX Levels (from 035f55f4)
# dex_levels is from 035f55f4
chart5_dex_levels_data = []
for i in range(min(6, len(dex_levels))):
    chart5_dex_levels_data.append({
        'strike': dex_levels.index[i],
        'call_dex': dex_levels.iloc[i]['CallDEX'] / 10**6, # Convert to Millions
        'put_dex': dex_levels.iloc[i]['PutDEX'] / 10**6   # Convert to Millions
    })


# CHART 6 - Delta Profile points (from 035f55f4)
# delta_at_spot_exfri, max_delta_positive_strike_exfri, max_delta_positive_value_exfri,
# min_delta_negative_strike_exfri, min_delta_negative_value_exfri are from 035f55f4

# Delta Flip from Chart 6 (Delta at Spot Price for Ex-Next Monthly Expiry line, labeled as Delta Flip OI)
chart6_delta_flip_oi_strike = spotPrice # Delta Flip strike is the Spot Price
chart6_delta_flip_oi_value = delta_at_spot_exfri # Delta at Spot Price value

# Max Delta Positivo (Ex-Next Monthly Expiry)
chart6_max_pos_strike = max_delta_positive_strike_exfri
chart6_max_pos_value = max_delta_positive_value_exfri # Already in Millions

# Min Delta Negativo (Ex-Next Monthly Expiry)
chart6_min_neg_strike = min_delta_negative_strike_exfri
chart6_min_neg_value = min_delta_negative_value_exfri # Already in Millions


# ==================== GERAÇÃO DA LINHA ÚNICA ====================

# Criar a string com todos os dados separados por vírgula
# Order: General, Chart 1, Chart 2, Chart 3, Chart 4, Chart 5, Chart 6

# General Data (3 values + date)
data_string = f"{spot_price:.2f},{total_gamma:.2f},{total_delta:.2f},{update_date},"

# Chart 1 Data
# Top 9 Neg Gamma (9 strikes * 2 values)
for item in chart1_neg_gamma_data:
    data_string += f"{item['gamma']:.4f},{item['strike']:.0f},"
# Top 9 Pos Gamma (9 strikes * 2 values)
for item in chart1_pos_gamma_data:
     data_string += f"{item['gamma']:.4f},{item['strike']:.0f},"
# Zero Gamma
data_string += f"{chart1_zero_gamma_strike:.0f},"
# OI/Vol Walls (4 walls * 2 values)
data_string += f"{chart1_call_oi_wall_value:.0f},{chart1_call_oi_wall_strike:.0f},"
data_string += f"{chart1_put_oi_wall_value:.0f},{chart1_put_oi_wall_strike:.0f},"
data_string += f"{chart1_call_vol_wall_value:.0f},{chart1_call_vol_wall_strike:.0f},"
data_string += f"{chart1_put_vol_wall_value:.0f},{chart1_put_vol_wall_strike:.0f},"
# Top 5 Calls OI (5 strikes * 2 values)
for item in chart1_top5_call_oi_data:
    data_string += f"{item['StrikePrice']:.0f},{item['CallOpenInt']:.0f},"
# Top 5 Puts OI (5 strikes * 2 values)
for item in chart1_top5_put_oi_data:
    data_string += f"{item['StrikePrice']:.0f},{item['PutOpenInt']:.0f},"
# Largest Volume and Open Interest (4 values * 2 = 8 values)
data_string += f"{chart1_largest_call_vol_strike:.0f},{chart1_largest_call_vol_value:.0f},"
data_string += f"{chart1_largest_put_vol_strike:.0f},{chart1_largest_put_vol_value:.0f},"
data_string += f"{chart1_largest_call_oi_strike:.0f},{chart1_largest_call_oi_value:.0f},"
data_string += f"{chart1_largest_put_oi_strike:.0f},{chart1_largest_put_oi_value:.0f},"


# Chart 2 GEX Levels (6 levels * 3 values)
for item in chart2_gex_levels_data:
    data_string += f"{item['strike']:.0f},{item['call_gex']:.4f},{item['put_gex']:.4f},"

# Chart 3 Gamma Profile points and Delta Flip (Estrutura)
# Gamma Profile points (4 points * 2 values)
data_string += f"{chart3_gamma_flip_value:.4f},{chart3_gamma_flip_strike:.0f},"
data_string += f"{chart3_vol_trigger_value:.4f},{chart3_vol_trigger_strike:.0f},"
data_string += f"{chart3_max_pos_value:.4f},{chart3_max_pos_strike:.0f},"
data_string += f"{chart3_min_neg_value:.4f},{chart3_min_neg_strike:.0f},"
# Delta Flip (Estrutura) from Chart 3
# Use the calculated delta at spot price for Ex-Next Monthly Expiry
data_string += f"{chart3_delta_flip_estrutura_value:.4f},{chart3_delta_flip_estrutura_strike:.0f},"


# Chart 4 Delta Data
# Top 9 Neg Delta (9 strikes * 2 values)
for item in chart4_neg_delta_data:
    data_string += f"{item['delta']:.4f},{item['strike']:.0f},"
# Top 9 Pos Delta (9 strikes * 2 values)
for item in chart4_pos_delta_data:
    data_string += f"{item['delta']:.4f},{item['strike']:.0f},"
# Zero Delta
data_string += f"{chart4_zero_delta_strike:.0f},"


# Chart 5 DEX Levels (6 levels * 3 values)
for item in chart5_dex_levels_data:
    data_string += f"{item['strike']:.0f},{item['call_dex']:.4f},{item['put_dex']:.4f},"

# Chart 6 Delta Profile points and Delta Flip (OI)
# Delta Flip (OI) from Chart 6 (Delta at Spot Price)
data_string += f"{chart6_delta_flip_oi_value:.4f},{chart6_delta_flip_oi_strike:.0f},"
# Max Delta Positivo (Ex-Next Monthly Expiry)
data_string += f"{chart6_max_pos_value:.4f},{chart6_max_pos_strike:.0f},"
# Min Delta Negativo (Ex-Next Monthly Expiry)
# Ensure the last value does not have a trailing comma
data_string += f"{chart6_min_neg_value:.4f},{chart6_min_neg_strike:.0f}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador 'SR Gamma/Delta Flip - COMPLETO'") # Updated indicator name
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS:\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL GAMMA: ${total_gamma:.2f} Bn")
print(f"📊 TOTAL DELTA: ${total_delta:.2f} M") # Added Total Delta
print(f"📅 DATA: {update_date}")

print("\n--- CHART 1: GAMMA EXPOSURE ---")
print("\n🔴 Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
for item in chart1_neg_gamma_data:
    print(f"  Strike {item['strike']:.0f}: Gamma: {item['gamma']:.4f} Bn")

print("\n🟢 Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
for item in chart1_pos_gamma_data:
    print(f"  Strike {item['strike']:.0f}: Gamma: {item['gamma']:.4f} Bn")

print(f"\n🟠 Zero Gamma (Gamma Flip): {chart1_zero_gamma_strike:.0f}")

print("\n📊 Paredes por Open Interest:")
print(f"  Call Wall (OI): {chart1_call_oi_wall_value:.0f} at strike {chart1_call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {chart1_put_oi_wall_value:.0f} at strike {chart1_put_oi_wall_strike:.0f}")

print("\n📊 Paredes por Volume:")
print(f"  Call Wall (Vol): {chart1_call_vol_wall_value:.0f} at strike {chart1_call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {chart1_put_vol_wall_value:.0f} at strike {chart1_put_vol_wall_strike:.0f}")

print("\n📈 Top 5 Calls por Open Interest:")
for item in chart1_top5_call_oi_data:
    print(f"  Strike {item['StrikePrice']:.0f}: {item['CallOpenInt']:.0f}")

print("\n📉 Top 5 Puts por Open Interest:")
for item in chart1_top5_put_oi_data:
    print(f"  Strike {item['StrikePrice']:.0f}: {item['PutOpenInt']:.0f}")

print("\n🏆 Strikes com Maior Volume e Open Interest (Geral):")
print(f"  Maior Volume de Call: {chart1_largest_call_vol_value:.0f} at strike {chart1_largest_call_vol_strike:.0f}")
print(f"  Maior Volume de Put: {chart1_largest_put_vol_value:.0f} at strike {chart1_largest_put_vol_strike:.0f}")
print(f"  Maior Open Interest de Call: {chart1_largest_call_oi_value:.0f} at strike {chart1_largest_call_oi_strike:.0f}")
print(f"  Maior Open Interest de Put: {chart1_largest_put_oi_value:.0f} at strike {chart1_largest_put_oi_strike:.0f}")


print("\n--- CHART 2: GEX LEVELS ---")
print("\n💎 Top 6 GEX Levels:")
for i, item in enumerate(chart2_gex_levels_data):
    net_gex = item['call_gex'] + item['put_gex']
    print(f"   {i+1}. Strike {item['strike']:.0f}: Net GEX = {net_gex:.2f} Bn (Call GEX: {item['call_gex']:.4f} Bn, Put GEX: {item['put_gex']:.4f} Bn)")

print("\n--- CHART 3: GAMMA PROFILE POINTS ---")
print("\n🟠 NÍVEIS ESPECIAIS (GAMMA PROFILE):")
print(f"   • Gamma Flip (Ex-Next Monthly): {chart3_gamma_flip_strike:.0f} (γ: {chart3_gamma_flip_value:.4f} Bn)")
print(f"   • Vol Trigger (Gamma Flip Point): {chart3_vol_trigger_strike:.0f} (γ: {chart3_vol_trigger_value:.4f} Bn)")
print(f"   • Max Gamma Pos (Ex-Next Monthly): {chart3_max_pos_strike:.0f} (γ: {chart3_max_pos_value:.4f} Bn)")
print(f"   • Min Gamma Neg (Ex-Next Monthly): {chart3_min_neg_strike:.0f} (γ: {chart3_min_neg_value:.4f} Bn)")
# Print Delta Flip (Estrutura) data using the calculated values
print(f"   • Delta Flip (Estrutura): {chart3_delta_flip_estrutura_strike:.0f} (Δ: {chart3_delta_flip_estrutura_value:.4f} M)")


print("\n--- CHART 4: DELTA EXPOSURE ---")
print("\n🔴 Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for item in chart4_neg_delta_data:
    print(f"  Strike {item['strike']:.0f}: Delta: {item['delta']:.4f} M")

print("\n🟢 Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for item in chart4_pos_delta_data:
    print(f"  Strike {item['strike']:.0f}: Delta: {item['delta']:.4f} M")

if chart4_zero_delta_strike != 0:
    print(f"\n🟠 Zero Delta (Delta Flip - Zero Cross): {chart4_zero_delta_strike:.0f}")
else:
    print("\n🟠 Zero Delta (Delta Flip - Zero Cross): Não encontrado")


print("\n--- CHART 5: DEX LEVELS ---")
print("\n💎 Top 6 DEX Levels:")
for i, item in enumerate(chart5_dex_levels_data):
    net_dex = item['call_dex'] + item['put_dex']
    print(f"   {i+1}. Strike {item['strike']:.0f}: Net DEX = {net_dex:.2f} M (Call DEX: {item['call_dex']:.4f} M, Put DEX: {item['put_dex']:.4f} M)")

print("\n--- CHART 6: DELTA PROFILE POINTS ---")
print("\n🟠 NÍVEIS ESPECIAIS (DELTA PROFILE):")
print(f"   • Delta Flip (OI): {chart6_delta_flip_oi_strike:.0f} (Δ: {chart6_delta_flip_oi_value:.4f} M)")
print(f"   • Max Delta Pos (Ex-Next Monthly): {chart6_max_pos_strike:.0f} (Δ: {chart6_max_pos_value:.4f} M)")
print(f"   • Min Delta Neg (Ex-Next Monthly): {chart6_min_neg_strike:.0f} (Δ: {chart6_min_neg_value:.4f} M)")

# Summary of Regimes and Distance
regime_gamma = "POSITIVE GAMMA ✅" if spot_price > chart3_gamma_flip_strike else "NEGATIVE GAMMA ⚠️"
print(f"\n📈 REGIME (GAMMA): {regime_gamma}")

distance_to_gamma_flip = ((spot_price - chart3_gamma_flip_strike) / chart3_gamma_flip_strike) * 100
print(f"📏 DISTÂNCIA DO FLIP (GAMMA): {distance_to_gamma_flip:.2f}%")

regime_delta = "POSITIVE DELTA ✅" if chart6_delta_flip_oi_value > 0 else "NEGATIVE DELTA ⚠️"
print(f"\n📈 REGIME (DELTA @ SPOT): {regime_delta}")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("✅ DADOS PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO ====================

print("🔍 VALIDAÇÃO DOS DADOS:\n")

# Verifica se há dados válidos
errors = []

if spot_price <= 0:
    errors.append("❌ Spot Price inválido")

# Basic checks for Chart 1 Gamma data
if not chart1_neg_gamma_data:
    errors.append("❌ Dados de Top 9 Gamma Negativo (Chart 1) não coletados")
if not chart1_pos_gamma_data:
    errors.append("❌ Dados de Top 9 Gamma Positivo (Chart 1) não coletados")
if chart1_zero_gamma_strike <= 0:
    errors.append("❌ Zero Gamma (Chart 1) inválido")
if chart1_call_oi_wall_strike <= 0 or chart1_put_oi_wall_strike <= 0 or chart1_call_vol_wall_strike <= 0 or chart1_put_vol_wall_strike <= 0:
     errors.append("❌ Strikes de Paredes OI/Volume (Chart 1) inválidos")

# Basic checks for Chart 2 GEX Levels
if not chart2_gex_levels_data:
    errors.append("❌ Dados do Chart 2 (GEX Levels) não coletados")

# Basic checks for Chart 3 Gamma Profile
if chart3_gamma_flip_strike <= 0:
    errors.append("❌ Gamma Flip Strike (Chart 3) inválido")
if chart3_vol_trigger_strike <= 0:
     errors.append("❌ Vol Trigger Strike (Chart 3) inválido")
if chart3_max_pos_strike <= 0 or chart3_min_neg_strike <= 0:
     errors.append("❌ Strikes de Max/Min Gamma (Chart 3) inválidos")
# Check for Chart 3 Delta Flip (Estrutura)
# Now it's always calculated at spot, so just check value range if needed
if chart3_delta_flip_estrutura_strike <= 0:
     errors.append("❌ Delta Flip (Estrutura) Strike (Chart 3) inválido")


# Basic checks for Chart 4 Delta Exposure
if not chart4_neg_delta_data:
    errors.append("❌ Dados de Top 9 Delta Negativo (Chart 4) não coletados")
if not chart4_pos_delta_data:
    errors.append("❌ Dados de Top 9 Delta Positivo (Chart 4) não coletados")
# chart4_zero_delta_strike can be 0 if not found, check against a reasonable range or None if applicable


# Basic checks for Chart 5 DEX Levels
if not chart5_dex_levels_data:
    errors.append("❌ Dados do Chart 5 (DEX Levels) não coletados")

# Basic checks for Chart 6 Delta Profile
if chart6_delta_flip_oi_strike <= 0: # This is spot price, already checked
     errors.append("❌ Delta Flip OI Strike (Chart 6) inválido")
if chart6_max_pos_strike <= 0 or chart6_min_neg_strike <= 0:
     errors.append("❌ Strikes de Max/Min Delta (Chart 6) inválidos")


# Check if required variables for data string were defined
required_vars = [
    'spot_price', 'total_gamma', 'total_delta', 'update_date',
    'chart1_neg_gamma_data', 'chart1_pos_gamma_data', 'chart1_zero_gamma_strike',
    'chart1_call_oi_wall_value', 'chart1_call_oi_wall_strike', 'chart1_put_oi_wall_value', 'chart1_put_oi_wall_strike',
    'chart1_call_vol_wall_value', 'chart1_call_vol_wall_strike', 'chart1_put_vol_wall_value', 'chart1_put_vol_wall_strike',
    'chart1_top5_call_oi_data', 'chart1_top5_put_oi_data',
    'chart1_largest_call_vol_strike', 'chart1_largest_call_vol_value', 'chart1_largest_put_vol_strike', 'chart1_largest_put_vol_value',
    'chart1_largest_call_oi_strike', 'chart1_largest_call_oi_value', 'chart1_largest_put_oi_strike', 'chart1_largest_put_oi_value',
    'chart2_gex_levels_data',
    'chart3_gamma_flip_strike', 'chart3_gamma_flip_value', 'chart3_vol_trigger_strike', 'chart3_vol_trigger_value',
    'chart3_max_pos_strike', 'chart3_max_pos_value', 'chart3_min_neg_strike', 'chart3_min_neg_value',
    'chart3_delta_flip_estrutura_strike', 'chart3_delta_flip_estrutura_value',
    'chart4_neg_delta_data', 'chart4_pos_delta_data', 'chart4_zero_delta_strike',
    'chart5_dex_levels_data',
    'chart6_delta_flip_oi_strike', 'chart6_delta_flip_oi_value', 'chart6_max_pos_strike', 'chart6_max_pos_value',
    'chart6_min_neg_strike', 'chart6_min_neg_value'
]

# This check is more robust than just checking if the variable is defined in the current scope,
# as it ensures the data structures are not empty if they are expected to contain data.
# This still won't catch NameErrors if a previous cell failed to define a variable used here.

if errors:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")
# ==================== CÓDIGO TRADINGVIEW (INPUTS) ====================
print("📋 CÓDIGO PARA TRADINGVIEW - COPIE E COLE NOS INPUTS\n")
print("="*80)
print("// Cole este código nos inputs do indicador TradingView")
print("// Substitua apenas os VALORES dos inputs existentes")
print("="*80 + "\n")

# DADOS GERAIS
print("// ==================== DADOS GERAIS ====================")
print(f"spotPrice = {spot_price:.2f}")
print(f"totalGamma = {total_gamma:.2f}")
print(f"totalDelta = {total_delta:.2f}")
print(f'updateDate = timestamp("{update_date}")')
print()

# CHART 1 - GAMMA EXPOSURE DATA
print("// ==================== CHART 1: GAMMA EXPOSURE ====================")
print("// Top 9 Strikes por Gamma Exposure Negativo")
for i, item in enumerate(chart1_neg_gamma_data):
    print(f"c1_neg_gamma_strike_{i+1} = {item['strike']:.0f}")
    print(f"c1_neg_gamma_value_{i+1} = {item['gamma']:.4f}")
print()

print("// Top 9 Strikes por Gamma Exposure Positivo")
for i, item in enumerate(chart1_pos_gamma_data):
    print(f"c1_pos_gamma_strike_{i+1} = {item['strike']:.0f}")
    print(f"c1_pos_gamma_value_{i+1} = {item['gamma']:.4f}")
print()

print("// Zero Gamma")
print(f"c1_zero_gamma_strike = {chart1_zero_gamma_strike:.0f}")
print()

print("// Paredes por Open Interest")
print(f"c1_call_oi_wall_value = {chart1_call_oi_wall_value:.0f}")
print(f"c1_call_oi_wall_strike = {chart1_call_oi_wall_strike:.0f}")
print(f"c1_put_oi_wall_value = {chart1_put_oi_wall_value:.0f}")
print(f"c1_put_oi_wall_strike = {chart1_put_oi_wall_strike:.0f}")
print()

print("// Paredes por Volume")
print(f"c1_call_vol_wall_value = {chart1_call_vol_wall_value:.0f}")
print(f"c1_call_vol_wall_strike = {chart1_call_vol_wall_strike:.0f}")
print(f"c1_put_vol_wall_value = {chart1_put_vol_wall_value:.0f}")
print(f"c1_put_vol_wall_strike = {chart1_put_vol_wall_strike:.0f}")
print()

print("// Top 5 Calls por Open Interest")
for i, item in enumerate(chart1_top5_call_oi_data):
    print(f"c1_top5_call_oi_strike_{i+1} = {item['StrikePrice']:.0f}")
    print(f"c1_top5_call_oi_value_{i+1} = {item['CallOpenInt']:.0f}")
print()

print("// Top 5 Puts por Open Interest")
for i, item in enumerate(chart1_top5_put_oi_data):
    print(f"c1_top5_put_oi_strike_{i+1} = {item['StrikePrice']:.0f}")
    print(f"c1_top5_put_oi_value_{i+1} = {item['PutOpenInt']:.0f}")
print()

print("// Strikes com Maior Volume e Open Interest (Geral)")
print(f"c1_largest_call_vol_strike = {chart1_largest_call_vol_strike:.0f}")
print(f"c1_largest_call_vol_value = {chart1_largest_call_vol_value:.0f}")
print(f"c1_largest_put_vol_strike = {chart1_largest_put_vol_strike:.0f}")
print(f"c1_largest_put_vol_value = {chart1_largest_put_vol_value:.0f}")
print(f"c1_largest_call_oi_strike = {chart1_largest_call_oi_strike:.0f}")
print(f"c1_largest_call_oi_value = {chart1_largest_call_oi_value:.0f}")
print(f"c1_largest_put_oi_strike = {chart1_largest_put_oi_strike:.0f}")
print(f"c1_largest_put_oi_value = {chart1_largest_put_oi_value:.0f}")
print()


# CHART 2 - GEX LEVELS
print("// ==================== CHART 2: GEX LEVELS ====================")
for i, item in enumerate(chart2_gex_levels_data):
    print(f"c2_gex_strike_{i+1} = {item['strike']:.0f}")
    print(f"c2_gex_call_{i+1} = {item['call_gex']:.4f}")
    print(f"c2_gex_put_{i+1} = {item['put_gex']:.4f}")
print()


# CHART 3 - GAMMA PROFILE POINTS
print("// ==================== CHART 3: GAMMA PROFILE POINTS ====================")
print(f"c3_gamma_flip_strike = {chart3_gamma_flip_strike:.0f}")
print(f"c3_gamma_flip_value = {chart3_gamma_flip_value:.4f}")
print()
print(f"c3_vol_trigger_strike = {chart3_vol_trigger_strike:.0f}")
print(f"c3_vol_trigger_value = {chart3_vol_trigger_value:.4f}")
print()
print(f"c3_max_pos_strike = {chart3_max_pos_strike:.0f}")
print(f"c3_max_pos_value = {chart3_max_pos_value:.4f}")
print()
print(f"c3_min_neg_strike = {chart3_min_neg_strike:.0f}")
print(f"c3_min_neg_value = {chart3_min_neg_value:.4f}")
print()
print("// Delta Flip (Estrutura) - Delta at Spot Price for Ex-Next Monthly Expiry")
print(f"c3_delta_flip_estrutura_strike = {chart3_delta_flip_estrutura_strike:.0f}")
print(f"c3_delta_flip_estrutura_value = {chart3_delta_flip_estrutura_value:.4f}")
print()


# CHART 4 - DELTA EXPOSURE DATA
print("// ==================== CHART 4: DELTA EXPOSURE ====================")
print("// Top 9 Strikes por Delta Exposure Negativo")
for i, item in enumerate(chart4_neg_delta_data):
    print(f"c4_neg_delta_strike_{i+1} = {item['strike']:.0f}")
    print(f"c4_neg_delta_value_{i+1} = {item['delta']:.4f}")
print()

print("// Top 9 Strikes por Delta Exposure Positivo")
for i, item in enumerate(chart4_pos_delta_data):
    print(f"c4_pos_delta_strike_{i+1} = {item['strike']:.0f}")
    print(f"c4_pos_delta_value_{i+1} = {item['delta']:.4f}")
print()

print("// Zero Delta")
print(f"c4_zero_delta_strike = {chart4_zero_delta_strike:.0f}")
print()

# CHART 5 - DEX LEVELS
print("// ==================== CHART 5: DEX LEVELS ====================")
for i, item in enumerate(chart5_dex_levels_data):
    print(f"c5_dex_strike_{i+1} = {item['strike']:.0f}")
    print(f"c5_dex_call_{i+1} = {item['call_dex']:.4f}")
    print(f"c5_dex_put_{i+1} = {item['put_dex']:.4f}")
print()


# CHART 6 - DELTA PROFILE POINTS
print("// ==================== CHART 6: DELTA PROFILE POINTS ====================")
print("// Delta Flip (OI) - Delta at Spot Price for Ex-Next Monthly Expiry")
print(f"c6_delta_flip_oi_strike = {chart6_delta_flip_oi_strike:.0f}")
print(f"c6_delta_flip_oi_value = {chart6_delta_flip_oi_value:.4f}")
print()
print(f"c6_max_pos_strike = {chart6_max_pos_strike:.0f}")
print(f"c6_max_pos_value = {chart6_max_pos_value:.4f}")
print()
print(f"c6_min_neg_strike = {chart6_min_neg_strike:.0f}")
print(f"c6_min_neg_value = {chart6_min_neg_value:.4f}")
print()

print("\n" + "="*80)
print("✅ CÓDIGO GERADO COM SUCESSO!")
print("="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA/DELTA FLIP (VERSÃO COMPLETA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

6851.14,65.28,152744.95,27 Oct 2025 00:00,-0.1897,6680,-0.2141,6550,-0.2303,6500,-0.2796,6675,-0.2976,6730,-0.3153,6700,-0.3208,6720,-0.3801,6740,-0.4185,6600,7.5771,6850,7.0814,6875,6.4448,6860,4.6002,6825,4.4433,6870,3.8921,6880,3.4717,6900,3.4222,6830,2.8613,6840,6762,23106,7000,37042,5420,39010,6880,19313,6850,7000,23106,7200,21357,6950,19314,6940,12891,6875,12382,5420,37042,5320,36847,6000,28212,5100,23599,5500,23099,6880,39010,6850,19313,7000,23106,5420,37042,6850,7.8293,-0.2522,6875,7.2299,-0.1485,6860,6.7913,-0.3465,6825,4.8768,-0.2767,6800,3.6859,-1.2853,6870,4.5108,-0.0675,7.6397,6781,0.0000,6762,58.7365,6874,-30.8880,6549,152744.9544,6851,-8.9770,6310,-10.0978,5320,-10.4348,5420,-11.6484,6555,-16.9877,6170,-19.0456,6085,-23.1633,6185,-159.9683,7025,-4711.5253,7000,9988.3179,6800,9715.6915,6750,8673.4106,6650,7497.5661,6850,7226.1410,6000,6826.2678,6